In [12]:
from datasets import load_dataset
import re
import jieba
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader

In [13]:
CACHE_DIR = "./dataset_cache"

dataset = load_dataset("jedwang/bert-base-split-chinese", split="train[:5%]",cache_dir=CACHE_DIR)
dataset = dataset.train_test_split(test_size = 0.1,seed=42)
train_dataset = [d["text"] for d in dataset["train"]]
test_dataset = [d["text"] for d in dataset["test"]]

In [14]:



def clean_text(text):
    text = re.sub(r"[^\u4e00-\u9fff]", "", text)
    return text.strip()

def tokenize(text):
    text = clean_text(text)
    return list(jieba.cut(text))

In [15]:
train_tokens = []
for t in train_dataset:
    tokens = tokenize(t)
    if 5 < len(tokens):
        train_tokens.append(tokens)

test_tokens = []
for t in test_dataset:
    tokens = tokenize(t)
    if 5 < len(tokens):
        test_tokens.append(tokens)

In [16]:
vocab = set()
for tokens in train_tokens:
    vocab.update(tokens)

vocab = list(vocab)
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for i,w in enumerate(vocab)}
vocab_size = len(vocab)

In [17]:
window_size = 2

#tokens_list保存多个list，每个list保存了一个句子分词后，每个词的id
def build_cbow_data(tokens_list,word2idx,window = 2):
    data = []
    for tokens in tokens_list:
        for i in range(window,len(tokens) - window):
            target = tokens[i]
            context = tokens[i - window:i] +tokens[i + 1:i + window + 1]

            #将context与target转为id，存入数据集
            context_idx = [word2idx[w] for w in context if w in word2idx]
            target_idx = word2idx[target] if target in word2idx else -1
            if len(context_idx) == 2*window and target != -1:
                data.append((context_idx,target_idx))
    return data

In [18]:
train_data = build_cbow_data(train_tokens, word2idx, window_size)
test_data = build_cbow_data(test_tokens, word2idx, window_size)

In [45]:

for i in train_data[0][0]:
  print(idx2word[i])
print(idx2word[train_data[0][1]])
#context
#死亡
#之河
#了
#两个

#target
#讲述

死亡
之河
了
两个
讲述


In [23]:
class CBOWDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context,target = self.data[idx]
        return torch.LongTensor(context), torch.LongTensor([target])

train_loader = DataLoader(CBOWDataset(train_data),batch_size=256,shuffle=True,num_workers=2)
test_loader = DataLoader(CBOWDataset(test_data),batch_size=256,shuffle=True,num_workers=2)

In [24]:
class CBOW(nn.Module):
    def __init__(self, vocab_size,embed_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embed_size)
        self.fc = nn.Linear(embed_size,vocab_size)

    def forward(self,x):
        embed = self.embedding(x).mean(dim=1)
        out = self.fc(embed)
        return out

In [25]:
embed_size = 100
model = CBOW(vocab_size,embed_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 1e-3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

CBOW(
  (embedding): Embedding(94188, 100)
  (fc): Linear(in_features=100, out_features=94188, bias=True)
)

In [28]:
print("训练。。。")
for epoch in range(5):
    model.train()
    total_loss = 0
    for context,target in train_loader:
        context,target = context.to(device),target.to(device).squeeze()

        optimizer.zero_grad()
        output = model(context)
        loss = criterion(output,target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch{epoch + 1},训练损失:{total_loss/len(train_loader):.4f}")

训练。。。
Epoch1,训练损失:8.4085
Epoch2,训练损失:7.6637
Epoch3,训练损失:7.0661
Epoch4,训练损失:6.5519
Epoch5,训练损失:6.0966


In [29]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for context, target in test_loader:
        context, target = context.to(device), target.to(device).squeeze()
        output = model(context)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)

print(f"\n测试集准确率: {correct/total:.4f}")


测试集准确率: 0.0840


In [30]:
# ===================== 10. 近义词查询（看效果） =====================
def get_top_similar_words(word, top_k=5):
    if word not in word2idx:
        print(f"词「{word}」不在词表里")
        return

    model.eval()
    with torch.no_grad():
        # 获取该词的词向量
        word_idx = torch.tensor([word2idx[word]]).to(device)
        word_vec = model.embedding(word_idx)  # [1, embed_size]

        # 计算和所有词的余弦相似度
        all_vecs = model.embedding.weight  # [vocab_size, embed_size]

        # 余弦相似度
        similarity = torch.cosine_similarity(word_vec, all_vecs, dim=1)

        # 取最相似的 top_k+1（排除自己）
        top_indices = similarity.topk(top_k + 1).indices[1:].cpu().numpy()

    print(f"\n与「{word}」最相关的 {top_k} 个词：")
    for idx in top_indices:
        print(f"  - {idx2word[idx]}")

# 选几个你想查的词（中文，尽量选常见词）
test_words = ["中国", "北京", "科学", "文化", "学习", "发展"]
for w in test_words:
    get_top_similar_words(w, top_k=5)


与「中国」最相关的 5 个词：
  - 追尊
  - 孙过庭
  - 五廷伦
  - 世界华人
  - 始

与「北京」最相关的 5 个词：
  - 硬肿
  - 煮烂
  - 机械制造
  - 临淮镇
  - 瑶

与「科学」最相关的 5 个词：
  - 溅水
  - 主要参数
  - 少尉
  - 缘觉
  - 华夏

与「文化」最相关的 5 个词：
  - 乡土
  - 实边
  - 杨义臣
  - 饰朝美
  - 雯

与「学习」最相关的 5 个词：
  - 有蒙切
  - 清史
  - 展
  - 后加
  - 文学奖

与「发展」最相关的 5 个词：
  - 加尔帕
  - 工诗
  - 服务大局
  - 弄虚作假
  - 特聘


In [32]:
for context, target in test_loader:
    context, target = context.to(device), target.to(device).squeeze()
    break


In [34]:
context.shape

torch.Size([256, 4])